# Data Exploration & Cleaning

In [2]:
import pandas as pd

# Load dataset
df = pd.read_csv("/content/sample_data/DiseaseAndSymptoms.csv")

# Basic info
df.head()

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Understand Dataset Structure

In [3]:
df.shape

(4920, 18)

In [4]:
df.columns

Index(['Disease', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4',
       'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9',
       'Symptom_10', 'Symptom_11', 'Symptom_12', 'Symptom_13', 'Symptom_14',
       'Symptom_15', 'Symptom_16', 'Symptom_17'],
      dtype='object')

### Preprocessing

In [5]:
# Check Missing Values
df.isnull().sum()

,0
Disease,0
Symptom_1,0
Symptom_2,0
Symptom_3,0
Symptom_4,348
Symptom_5,1206
Symptom_6,1986
Symptom_7,2652
Symptom_8,2976
Symptom_9,3228


In [6]:
# Fill Missing Values Safely
df = df.fillna("")

In [7]:
# Combine all symptom columns into one
symptom_cols = df.columns[1:]

df["combined_symptoms"] = df[symptom_cols].apply(
    lambda row: " ".join(row.values.astype(str)), axis=1
)

In [8]:
# Clean Symptom Text
import re

def clean_text(text):
    text = text.lower()
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["combined_symptoms"] = df["combined_symptoms"].apply(clean_text)

In [9]:
df[["Disease", "combined_symptoms"]].head()

,Disease,combined_symptoms
0,Fungal infection,itching skin rash nodal skin eruptions dischro...
1,Fungal infection,skin rash nodal skin eruptions dischromic patches
2,Fungal infection,itching nodal skin eruptions dischromic patches
3,Fungal infection,itching skin rash dischromic patches
4,Fungal infection,itching skin rash nodal skin eruptions


In [10]:
# Encode Disease Labels
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["disease_encoded"] = le.fit_transform(df["Disease"])

In [11]:
# Save Cleaned Dataset
final_df = df[["combined_symptoms", "disease_encoded"]]

final_df.to_csv(
    "/content/sample_data/Encoded_DiseaseAndSymptoms.csv",
    index=False
)

# Disease Prediction Model

In [12]:
import pandas as pd

# Load cleaned dataset
df = pd.read_csv("/content/sample_data/Encoded_DiseaseAndSymptoms.csv")

df.head()


,combined_symptoms,disease_encoded
0,itching skin rash nodal skin eruptions dischro...,15
1,skin rash nodal skin eruptions dischromic patches,15
2,itching nodal skin eruptions dischromic patches,15
3,itching skin rash dischromic patches,15
4,itching skin rash nodal skin eruptions,15


In [13]:
# Train-Test Split
from sklearn.model_selection import train_test_split

X = df["combined_symptoms"]
y = df["disease_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### Feature Engineering

In [14]:
# Convert Text → Numbers (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [15]:
# Train the ML Model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=2000,
    multi_class="auto"
)

model.fit(X_train_tfidf, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(max_iter=2000, multi_class='auto')

In [16]:
# Evaluate the Model
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print(classification_report(y_test, y_pred))

Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24
           1       1.00      1.00      1.00        24
           2       1.00      1.00      1.00        24
           3       1.00      1.00      1.00        24
           4       1.00      1.00      1.00        24
           5       1.00      1.00      1.00        24
           6       1.00      1.00      1.00        24
           7       1.00      1.00      1.00        24
           8       1.00      1.00      1.00        24
           9       1.00      1.00      1.00        24
          10       1.00      1.00      1.00        24
          11       1.00      1.00      1.00        24
          12       1.00      1.00      1.00        24
          13       1.00      1.00      1.00        24
          14       1.00      1.00      1.00        24
          15       1.00      1.00      1.00        24
          16       1.00      1.00      1.00        24
          17 

### Model Deployment

In [19]:
# Save the Trained Model
import joblib

joblib.dump(model, "/content/model/disease_model.pkl")
joblib.dump(tfidf, "/content/model/tfidf_vectorizer.pkl")

['/content/model/tfidf_vectorizer.pkl']

In [20]:
# Save Label Encoder
joblib.dump(le, "/content/model/label_encoder.pkl")

['/content/model/label_encoder.pkl']

In [21]:
df.duplicated(subset=["combined_symptoms"]).sum()

np.int64(4616)

In [22]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X_train_tfidf, y_train, cv=5)
print("CV Accuracy:", scores.mean())

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

CV Accuracy: 1.0


In [23]:
sample = ["fever headache nausea joint pain"]
sample_vec = tfidf.transform(sample)
prediction = model.predict(sample_vec)
le.inverse_transform(prediction)

array(['Dengue'], dtype=object)